# Video Game Sales Analysis Project
## Project Overview

## Environment Setup and Required Libraries

In [1]:
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import ipywidgets as widgets


## Step 1: Loading and Initial Data Exploration


In [2]:
games = pd.read_csv("games.csv")

In [3]:
games.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 2.0 MB


In [5]:
games.describe()

,Year_of_Release,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score
count,16446.000000,16715.000000,16715.000000,16715.000000,16715.000000,8137.000000
mean,2006.484616,0.263377,0.145060,0.077617,0.047342,68.967679
std,5.877050,0.813604,0.503339,0.308853,0.186731,13.938165
min,1980.000000,0.000000,0.000000,0.000000,0.000000,13.000000
25%,2003.000000,0.000000,0.000000,0.000000,0.000000,60.000000
50%,2007.000000,0.080000,0.020000,0.000000,0.010000,71.000000
75%,2010.000000,0.240000,0.110000,0.040000,0.030000,79.000000
max,2016.000000,41.360000,28.960000,10.220000,10.570000,98.000000


In [6]:
games.duplicated().sum()

np.int64(0)

In [7]:
display(games["Name"].duplicated().sum())  # <-- we see duplicates in the Name column; let's check whether these are real duplicates
print("---------")

result = games.groupby("Name").agg(
                        count_of_name=("Name", "size"),
                        platforms=("Platform", lambda x: ", ".join(sorted(x.unique())))
).reset_index().sort_values('count_of_name', ascending=False)

result = result[result['count_of_name']>1]
result  # <-- from this we can conclude these are not real duplicates: the same game name was released on multiple platforms

np.int64(5155)

---------


,Name,count_of_name,platforms
6715,Need for Speed: Most Wanted,12,"DS, GBA, GC, PC, PS2, PS3, PSV, WiiU, X360, XB"
2952,FIFA 14,9,"3DS, PC, PS3, PS4, PSP, PSV, Wii, X360, XOne"
7785,Ratatouille,9,"DS, GBA, GC, PC, PS2, PS3, PSP, Wii, X360"
5143,LEGO Marvel Super Heroes,9,"3DS, DS, PC, PS3, PS4, PSV, WiiU, X360, XOne"
5470,Madden NFL 07,9,"DS, GBA, GC, PS2, PS3, PSP, Wii, X360, XB"
...,...,...,...
2837,Escape The Museum,2,"DS, Wii"
2846,Eternal Sonata,2,"PS3, X360"
2850,Etrian Odyssey II: Heroes of Lagaard,2,"3DS, DS"
2860,Evangelion: Jo,2,"PS2, PSP"


In [8]:
print(f"Total number of records : {len(games)}")
print(f"Total number of records : {games.shape[0]}")

Total number of records : 16715
Total number of records : 16715


In [9]:
pd.DataFrame({
    'column': games.columns,
    'dtype': games.dtypes.values,
    'null_count': games.isnull().sum().values,
    'unique_count': games.nunique().values
})


,column,dtype,null_count,unique_count
0,Name,str,2,11559
1,Platform,str,0,31
2,Year_of_Release,float64,269,37
3,Genre,str,2,12
4,NA_sales,float64,0,402
5,EU_sales,float64,0,307
6,JP_sales,float64,0,244
7,Other_sales,float64,0,155
8,Critic_Score,float64,8578,82
9,User_Score,str,6701,96


In [10]:
print(f'Null cells in critical_score :{round((games['Critic_Score'].isnull().sum()/games.shape[0])*100)}%') #< in critic_score column 51% of this column is empty
print(f'Null cells in user_score :{round((games['User_Score'].isnull().sum()/games.shape[0])*100)}%')  #< in user_score column 40% of this column is empty
print(f'Null cells in rating :{round((games['Rating'].isnull().sum()/games.shape[0])*100)}%')  #< in rating column 40% of this column is empty
print(f'Null cells in name :{round(games['Name'].isnull().sum())}')
print(f'Null cells in genre :{round(games['Genre'].isnull().sum())}')

Null cells in critical_score :51%
Null cells in user_score :40%
Null cells in rating :40%
Null cells in name :2
Null cells in genre :2


The dataset has 16,715 rows and 11 columns. `critic_score (~51%)`, `user_score (~40%)`, and `rating (~40%)` have substantial missing values. `name` and `genre` have only `2` missing values each. No duplicate rows were found.

The main issue is the high missingness in rating-related columns. These values should be handled carefully rather than simply removed, as they represent a large share of the dataset.

## Step 2: Data Preparation



### 2.1 Standardizing Column Names

In [11]:
games.columns = games.columns.str.lower()
games.columns.to_list()

['name',
 'platform',
 'year_of_release',
 'genre',
 'na_sales',
 'eu_sales',
 'jp_sales',
 'other_sales',
 'critic_score',
 'user_score',
 'rating']

In [12]:
games

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16710,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16711,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.00,0.00,NaN,NaN,NaN
16712,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16713,Spirits & Spells,GBA,2003.0,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN


### 2.2 Data Type Conversion

In [13]:
# Check current data types
games.dtypes

name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score             str
rating                 str
dtype: object

In [14]:
games[games['name'].isnull()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
659,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14244,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


From this, we can see that the `name` and `genre` columns contain missing values. Since these columns are important for the analysis, the remaining data in those rows is not useful, so I will remove them.


In [15]:
games = games.dropna(subset=['name', 'genre']).reset_index(drop=True)
print(f"Count of new rows: {games.shape[0]}")

Count of new rows: 16713


In [16]:
games['year_of_release']=games['year_of_release'].astype('Int64') 

print(games['year_of_release'].dtypes)

Int64


In [17]:
games['user_score'].value_counts().head()

user_score
tbd    2424
7.8     324
8       290
8.2     282
8.3     254
Name: count, dtype: int64

In [18]:
games['user_score'] = games['user_score'].replace('tbd',np.nan).astype('float64')
games['user_score'].dtypes

dtype('float64')

`tbd`: `user_score` contains `tbd` (“to be determined”), meaning no actual user rating is available yet.

`tbd` will be treated as `NaN`, not `0`, because `0` would incorrectly represent a very low rating.


In [19]:
games['user_score'].isna().sum()

np.int64(9123)



* `user_score`: `object` → `float64` because ratings are numeric and require mathematical operations.
* `year_of_release`: `float64` → `Int64` because years cannot be decimals, while `Int64` supports `NaN` values.


### 2.3 Handling Missing Values

In [20]:
games.isnull().sum()

name                  0
platform              0
year_of_release     269
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8576
user_score         9123
rating             6764
dtype: int64

As we can see, **269 rows have missing values in the `year` column**, so I decided to remove them.


In [21]:
games = games.dropna(subset=['year_of_release']).reset_index(drop=True)
print(f"count of new rows: {games.shape[0]}")

count of new rows: 16444


In [22]:
missing_values = pd.DataFrame({'null_count': games.isnull().sum(),  
                   'null_percent': round(games.isnull().sum()/len(games)*100,1)})

missing_values[missing_values['null_count']>0].sort_values('null_percent', ascending=False)

,null_count,null_percent
user_score,8981,54.6
critic_score,8461,51.5
rating,6676,40.6


In [23]:
pattern = games[['user_score','critic_score','rating']].isnull()
pattern.value_counts()[(True,True,True)]

np.int64(6580)

In [24]:
missing_count=pattern.all(axis=1).sum()
missing_count

np.int64(6580)

In [25]:
missing_by_year = games.groupby("year_of_release")[['user_score','critic_score','rating']].apply(lambda x: round(x.isnull().mean()*100))
missing_by_year

,user_score,critic_score,rating
year_of_release,,,
1980,100.0,100.0,100.0
1981,100.0,100.0,100.0
1982,100.0,100.0,100.0
1983,100.0,100.0,100.0
1984,100.0,100.0,100.0
1985,93.0,93.0,93.0
1986,100.0,100.0,100.0
1987,100.0,100.0,100.0
1988,93.0,93.0,93.0


In [26]:
games_1980_2000 = games[(games['year_of_release']>=1980)&(games['year_of_release']<2000)].count()
games_1980_2000

name               1974
platform           1974
year_of_release    1974
genre              1974
na_sales           1974
eu_sales           1974
jp_sales           1974
other_sales        1974
critic_score         96
user_score           92
rating              105
dtype: int64

In [27]:
games_2000_2016 = games[(games['year_of_release']>=2000)&(games['year_of_release']<=2016)].isnull().sum()
games_2000_2016

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       6583
user_score         7099
rating             4807
dtype: int64

* We can also see that almost none of the games released between 1980 and 2000 had ratings.
* We can also see from `games_1980_2000` that **1,974 games** were released between these years. Among them, **96** have a `critic_score`, **92** have a `user_score`, and **105** have a `rating`.
* `games_2000_2016`, **14,470 games** were released. Among them, **6,583** have missing `critic_score`, **7,099** have missing `user_score`, and **4,807** have missing `rating` values, meaning no rating was provided.
> As we can see from both cells above, there are **6,580** rows where all 3 columns (`user_score`, `critic_score`, `rating`) are empty (null).

In [28]:
percent  = round(missing_count/games.shape[0]*100)
percent

40

> I decided to keep these columns unchanged because the missing values, especially in these three columns, account for **40% or more of the data**. Therefore, I decided to preserve the missing values rather than impute them.


### 2.4 Calculate Total Sales

In [ ]:
games[['na_sales','eu_sales','jp_sales','other_sales']].sum()

na_sales       4341.42
eu_sales       2399.68
jp_sales       1290.64
other_sales     782.63
dtype: float64

In [30]:
games['total_sales']= games['na_sales'] + games['eu_sales'] + games['jp_sales'] + games['other_sales']
games[['name', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'total_sales']].head()

,name,na_sales,eu_sales,jp_sales,other_sales,total_sales
0,Wii Sports,41.36,28.96,3.77,8.45,82.54
1,Super Mario Bros.,29.08,3.58,6.81,0.77,40.24
2,Mario Kart Wii,15.68,12.76,3.79,3.29,35.52
3,Wii Sports Resort,15.61,10.93,3.28,2.95,32.77
4,Pokemon Red/Pokemon Blue,11.27,8.89,10.22,1.00,31.38


In [31]:
total_sales_overall = games['total_sales'].sum()
print(f"Total sales (all games, across all regions): {total_sales_overall:.2f} mln USD")

Total sales (all games, across all regions): 8814.37 mln USD


# Step 3: Analyzing Video Game Sales Data


## 3.1 Temporal Analysis of Game Releases

In [32]:
games_per_year = games.groupby('year_of_release').size().reset_index(name='games_count')
games_per_year


,year_of_release,games_count
0,1980,9
1,1981,46
2,1982,36
3,1983,17
4,1984,14
5,1985,14
6,1986,21
7,1987,16
8,1988,15
9,1989,17


In [33]:
fig = px.bar(games_per_year, x='year_of_release', y='games_count',
             color='games_count',
             title='Count of games by years',
             labels={'year_of_release': 'Year', 'games_count': 'Count of games'})
fig.show()

In [34]:
games_per_year['games_count'].describe()

count      37.000000
mean      444.432432
std       451.604334
min         9.000000
25%        36.000000
50%       338.000000
75%       762.000000
max      1427.000000
Name: games_count, dtype: float64

> Annual game releases were low from 1980–1993, increased sharply after 1994, and peaked in 2008–2009 at ~1,400+ games/year before gradually declining.
>The early years have limited statistical significance due to the small sample size. The 2012–2016 decline may reflect incomplete 2016 data and the exclusion of some mobile/digital sales from traditional sales statistics

## 3.2 Platform Sales Analysis Over Time

In [35]:
total_sales_by_year = games.groupby('platform')['total_sales'].sum().sort_values(ascending=False).round(1).reset_index()
top10_platforms = total_sales_by_year.head(10)
top10_platforms

top10_platforms.columns = ['platform', 'total_sales']

fig = px.bar(top10_platforms, x='platform', y='total_sales')
fig.show()

In [36]:
platform_year_sales = (games.groupby(['platform', 'year_of_release'], as_index=False)['total_sales'].sum().sort_values(['platform', 'year_of_release']).reset_index())

platform_year_sales

,index,platform,year_of_release,total_sales
0,0,2600,1980,11.38
1,1,2600,1981,35.68
2,2,2600,1982,28.88
3,3,2600,1983,5.84
4,4,2600,1984,0.27
...,...,...,...,...
233,233,XB,2008,0.18
234,234,XOne,2013,18.96
235,235,XOne,2014,54.07
236,236,XOne,2015,60.14


In [37]:
heatmap_data = platform_year_sales.pivot(
    index="platform",
    columns="year_of_release",
    values="total_sales"
).fillna(0)

fig_heatmap = px.imshow(
    heatmap_data,
    x=heatmap_data.columns.astype(str),
    y=heatmap_data.index,
    color_continuous_scale=px.colors.sequential.Plasma,
    labels={"x": "Year of Release", "y": "Platform", "color": "Total Sales"},
    title="Platform Sales Over Time"
)

fig_heatmap.update_layout(
    height=700,
    width=1200
)

fig_heatmap.show()


In [38]:

first_last = platform_year_sales.groupby('platform').agg(
    first_year=('year_of_release', 'min'),
    last_year=('year_of_release', 'max')
).reset_index()


def get_sales(row, year_col):
    year = row[year_col]
    platform = row['platform']
    sales = platform_year_sales[
        (platform_year_sales['platform'] == platform) & 
        (platform_year_sales['year_of_release'] == year)
    ]['total_sales'].values
    return sales[0] if len(sales) > 0 else 0

first_last['sales_first_year'] = first_last.apply(lambda r: get_sales(r, 'first_year'), axis=1)
first_last['sales_last_year'] = first_last.apply(lambda r: get_sales(r, 'last_year'), axis=1)


first_last['change'] = first_last['sales_last_year'] - first_last['sales_first_year']


top10_declining = first_last.sort_values('change').head(10)
top10_declining


,platform,first_year,last_year,sales_first_year,sales_last_year,change
26,Wii,2006,2016,137.15,0.18,-136.97
2,3DS,2011,2016,63.20,15.14,-48.06
16,PS2,2000,2011,39.17,0.45,-38.72
10,N64,1996,2002,34.10,0.08,-34.02
7,GC,2001,2007,26.34,0.27,-26.07
23,SNES,1990,1999,26.15,0.26,-25.89
17,PS3,2006,2016,20.96,3.60,-17.36
27,WiiU,2012,2016,17.56,4.60,-12.96
11,NES,1983,1994,10.96,0.11,-10.85
0,2600,1980,1989,11.38,0.63,-10.75


In [39]:
declining_platform_names = top10_declining['platform'].tolist()

declining_trend_data = platform_year_sales[platform_year_sales['platform'].isin(declining_platform_names)]

fig = px.line(declining_trend_data, x='year_of_release', y='total_sales', color='platform',
              title='Annual sales of the Top 10 declining platforms',
              labels={'year_of_release': 'year', 'total_sales': 'Sales', 'platform': 'Platform'},
              markers=True)
fig.show()

In [40]:
platform_active = games.groupby('platform')['year_of_release'].agg(first_year = 'min',last_year = 'max',games_released_in_active = 'count')
platform_active['platform_lifespan']= platform_active['last_year']-platform_active['first_year']
   # <--This analysis is not entirely accurate because our dataset only goes up to 2016. I wanted to identify which platforms are still active today, but this analysis can only show us which platforms had already become inactive by 2016.

max_year = games['year_of_release'].max()
platform_active['is_discontinued'] = platform_active['last_year'] < (max_year - 3)
discontinued_platforms = platform_active[platform_active['is_discontinued']].sort_values('last_year', ascending=False)
discontinued_platforms

,first_year,last_year,games_released_in_active,platform_lifespan,is_discontinued
platform,,,,,
PS2,2000,2011,2127,11,True
DC,1998,2008,52,10,True
XB,2000,2008,803,8,True
GBA,2000,2007,811,7,True
GC,2001,2007,542,6,True
PS,1994,2003,1190,9,True
N64,1996,2002,316,6,True
GB,1988,2001,97,13,True
WS,1999,2001,6,2,True


In [41]:
fig = px.timeline(
    discontinued_platforms.reset_index().assign(
        start=lambda d: pd.to_datetime(d['first_year'].astype(str) + '-01-01'),
        end=lambda d: pd.to_datetime(d['last_year'].astype(str) + '-12-31')
    ),
    x_start='start', x_end='end', y='platform',
    title='Active period of discontinued platforms',
    labels={'platform': 'Platform'},
    color='last_year', color_continuous_scale='Reds_r'
)
fig.update_yaxes(categoryorder='total ascending')
fig.show()

In [42]:
print(f"Median lifespan: {platform_active['platform_lifespan'].median()} years")
print(f"Mean lifespan: {platform_active['platform_lifespan'].mean():.1f} years")
print(f"Most common lifespan (mode): {platform_active['platform_lifespan'].mode().tolist()} years")

Median lifespan: 6.0 years
Mean lifespan: 7.6 years
Most common lifespan (mode): [11] years


Older platforms like  `PS2`, `DS`, and `GBA` reached zero sales after 2011–2013, while `PS4 and XOne` launched in 2013. The average platform lifespan is `~7–8 years`.

The `2013` launch of `PS4/XOne` marks a clear generation transition, with older platforms gradually declining over the following `2–3 years`.

## 3.3 Determining Relevant Time Period

In [43]:

relevant_years = [2013,2014,2015,2016]
games_relevant = games[games['year_of_release'].isin(relevant_years)]
games_relevant

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
16,Grand Theft Auto V,PS3,2013,Action,7.02,9.09,0.98,3.96,97.0,8.2,M,21.05
23,Grand Theft Auto V,X360,2013,Action,9.66,5.14,0.06,1.41,97.0,8.1,M,16.27
31,Call of Duty: Black Ops 3,PS4,2015,Shooter,6.03,5.86,0.36,2.38,NaN,NaN,NaN,14.63
33,Pokemon X/Pokemon Y,3DS,2013,Role-Playing,5.28,4.19,4.35,0.78,NaN,NaN,NaN,14.60
42,Grand Theft Auto V,PS4,2014,Action,3.96,6.31,0.38,1.97,97.0,8.3,M,12.62
...,...,...,...,...,...,...,...,...,...,...,...,...
16432,Strawberry Nauts,PSV,2016,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01
16436,Aiyoku no Eustia,PSV,2014,Misc,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01
16439,Samurai Warriors: Sanada Maru,PS3,2016,Action,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01
16441,Haitaka no Psychedelica,PSV,2016,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01


I decided to use 2013–2016 as the relevant years period.

* 2013 marked the launch of the new-generation PS4 and XOne, creating a major market shift.
* Older platforms such as PS2, PS3, X360, Wii, and DS were already declining or obsolete.
* Therefore, 2013–2016 better reflects the market conditions relevant for 2017 forecasting.
* Including 1980–2012 data could distort the forecast because obsolete platforms no longer provide meaningful future signals.

In [44]:
platform_yearly = games_relevant.groupby(['year_of_release', 'platform'])['total_sales'].sum().reset_index()

platform_2013 = platform_yearly[platform_yearly['year_of_release']==2013].set_index('platform')['total_sales']
platform_2016 = platform_yearly[platform_yearly['year_of_release']==2016].set_index('platform')['total_sales']

summary = pd.DataFrame({'sales_2013': platform_2013, 'sales_2016': platform_2016}).fillna(0)
summary['change'] = summary['sales_2016'] - summary['sales_2013']
summary = summary.sort_values('change', ascending=False)
summary

,sales_2013,sales_2016,change
platform,,,
PS4,25.99,69.25,43.26
XOne,18.96,26.15,7.19
DS,1.54,0.00,-1.54
PSP,3.14,0.00,-3.14
PSV,10.59,4.25,-6.34
PC,12.38,5.25,-7.13
Wii,8.59,0.18,-8.41
WiiU,21.65,4.60,-17.05
3DS,56.57,15.14,-41.43


In [45]:
def forecast_2017(group):
    if group['year_of_release'].nunique() < 2:
        return np.nan
    slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(
        group['year_of_release'], group['total_sales']
    )

    predicted = slope * 2017 + intercept
    return max(predicted, 0)  

platform_forecast = platform_yearly.groupby('platform').apply(forecast_2017, include_groups=False)
platform_forecast = platform_forecast.sort_values(ascending=False)
platform_forecast.head(10)

platform
PS4     115.705
XOne     46.740
PC        3.320
PSV       2.080
WiiU      1.950
3DS       0.745
PSP       0.000
PS3       0.000
Wii       0.000
X360      0.000
dtype: float64

In [46]:
top_platforms_2016 = platform_yearly[platform_yearly['year_of_release']==2016].nlargest(5, 'total_sales')['platform'].tolist()

comparison_data = []
for p in top_platforms_2016:
    hist = platform_yearly[platform_yearly['platform']==p][['year_of_release', 'total_sales']].copy()
    hist['type'] = 'Actual'
    hist['platform'] = p

    sales_2016 = hist[hist['year_of_release']==2016]['total_sales'].values[0]

    forecast_row = pd.DataFrame({
        'year_of_release': [2016, 2017],
        'total_sales': [sales_2016, platform_forecast[p]],
        'type': ['Forecast', 'Forecast']
    })
    forecast_row['platform'] = p

    comparison_data.append(pd.concat([hist, forecast_row]))

comparison_df = pd.concat(comparison_data)

fig = px.line(comparison_df, x='year_of_release', y='total_sales', color='platform', line_dash='type',
              title='2013-2016 actual sales + 2017 linear trend forecast',
              labels={'year_of_release':'Year', 'total_sales':'Sales (mln USD)', 'platform':'Platform'},
              markers=True)
fig.show()

## 3.4 Platform Performance Analysis

In [47]:
total_sales_by_year_relevant = games_relevant.groupby('platform')['total_sales'].agg('sum').sort_values(ascending = False)

fig_relevant = px.bar(total_sales_by_year_relevant.reset_index(),x='platform',y='total_sales',
                      title ='Total sales between 2013 and 2016',
                      labels={'platform': 'Platform', 'total_sales': 'Total sales'})
fig_relevant.show()

In [48]:
total_sales_by_year_relevant2 = games_relevant.groupby(['platform','year_of_release'])['total_sales'].agg('sum').reset_index()

line = px.line(total_sales_by_year_relevant2, x= 'year_of_release', y= 'total_sales', color= 'platform',
               title = 'Sales trend of platforms 2013-2016',
               labels={'year_of_release': 'Year', 'total_sales': 'Sales', 'platform': 'Platform'},
               markers=True) 

line.show()

In [49]:
total_sales_by_year_relevant = games_relevant.groupby('platform')['total_sales'].agg('sum').sort_values(ascending=False)
total_sales_by_year_relevant

platform
PS4     314.14
PS3     181.43
XOne    159.32
3DS     143.25
X360    136.80
WiiU     64.63
PC       39.43
PSV      32.99
Wii      13.66
PSP       3.50
DS        1.54
Name: total_sales, dtype: float64

In [50]:
total_sales_by_year_relevant2 = total_sales_by_year_relevant2.sort_values(['platform', 'year_of_release'])
total_sales_by_year_relevant2['yoy_growth_pct'] = total_sales_by_year_relevant2.groupby('platform')['total_sales'].pct_change() * 100
total_sales_by_year_relevant2


fig = px.bar(total_sales_by_year_relevant2.dropna(subset=['yoy_growth_pct']),
             x='year_of_release', y='yoy_growth_pct', color='platform', barmode='group',
             title='Year over year growth rate',
             labels={'year_of_release': 'year', 'yoy_growth_pct': 'YoY change %', 'platform': 'Platform'})
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.show()

## 3.5 Sales Distribution Analysis

In [51]:
fig_games = px.box(games, x='platform', y='total_sales',
             title='Distribution of sales by platform (All-time, 1980-2016)',
             labels={'platform': 'Platform', 'total_sales': 'Total sales by a game'})
fig_games.update_yaxes(range=[0, 4])
fig_games.show()

In [52]:
fig = px.box(games_relevant, x='platform', y='total_sales',
             title='Distribution of sales (2013-2016)',
             labels={'platform': 'Platform', 'total_sales': 'Total sales by a game'})
fig.update_yaxes(range=[0, 3])  # cap the y-axis to cut off extreme outliers and make the main distribution visible
fig.show()

In [53]:
games_relevant_stat = games_relevant.groupby('platform')['total_sales'].agg(['mean', 'median', 'std', 'count']).sort_values('mean', ascending=False).round(3)
games_relevant_stat

,mean,median,std,count
platform,,,,
PS4,0.801,0.200,1.609,392
X360,0.735,0.265,1.663,186
XOne,0.645,0.220,1.036,247
Wii,0.594,0.180,0.915,23
WiiU,0.562,0.200,1.039,115
PS3,0.526,0.150,1.452,345
3DS,0.473,0.090,1.381,303
PC,0.209,0.080,0.352,189
DS,0.192,0.150,0.172,8


> Median sales are low across most platforms (0.02–0.27M), shows a right-skewed, long-tail distribution driven by a few hit games. PS4 has the highest mean sales (0.80M) and standard deviation (1.61M).

> Platform differences are significant: PS4/X360 have stronger hit-game sales potential, while PSP/PSV show much lower sales, reflecting market decline.

## 3.6 Review Score Impact Analysis

In [54]:
ps4_critic = games_relevant[games_relevant['platform'] == 'PS4'].dropna(subset=['critic_score','total_sales'])
ps4_user = games_relevant[games_relevant['platform'] == 'PS4'].dropna(subset=['user_score','total_sales'])
display(ps4_critic)
display(ps4_user)


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
42,Grand Theft Auto V,PS4,2014,Action,3.96,6.31,0.38,1.97,97.0,8.3,M,12.62
77,FIFA 16,PS4,2015,Sports,1.12,6.12,0.06,1.28,82.0,4.3,E,8.58
92,Call of Duty: Advanced Warfare,PS4,2014,Shooter,2.81,3.48,0.14,1.23,83.0,5.7,M,7.66
94,FIFA 17,PS4,2016,Sports,0.66,5.75,0.08,1.11,85.0,5.0,E,7.60
105,Fallout 4,PS4,2015,Role-Playing,2.53,3.27,0.24,1.13,87.0,6.5,M,7.17
...,...,...,...,...,...,...,...,...,...,...,...,...
16216,Super Dungeon Bros,PS4,2016,Action,0.01,0.00,0.00,0.00,42.0,2.3,E10+,0.01
16229,Sherlock Holmes: The Devil's Daughter,PS4,2016,Adventure,0.01,0.00,0.00,0.00,70.0,6.8,T,0.01
16230,Root Letter,PS4,2016,Adventure,0.00,0.00,0.01,0.00,69.0,7.5,NaN,0.01
16255,Dungeons 2,PS4,2016,Role-Playing,0.01,0.00,0.00,0.00,61.0,7.9,T,0.01


,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
42,Grand Theft Auto V,PS4,2014,Action,3.96,6.31,0.38,1.97,97.0,8.3,M,12.62
77,FIFA 16,PS4,2015,Sports,1.12,6.12,0.06,1.28,82.0,4.3,E,8.58
92,Call of Duty: Advanced Warfare,PS4,2014,Shooter,2.81,3.48,0.14,1.23,83.0,5.7,M,7.66
94,FIFA 17,PS4,2016,Sports,0.66,5.75,0.08,1.11,85.0,5.0,E,7.60
105,Fallout 4,PS4,2015,Role-Playing,2.53,3.27,0.24,1.13,87.0,6.5,M,7.17
...,...,...,...,...,...,...,...,...,...,...,...,...
16216,Super Dungeon Bros,PS4,2016,Action,0.01,0.00,0.00,0.00,42.0,2.3,E10+,0.01
16229,Sherlock Holmes: The Devil's Daughter,PS4,2016,Adventure,0.01,0.00,0.00,0.00,70.0,6.8,T,0.01
16230,Root Letter,PS4,2016,Adventure,0.00,0.00,0.01,0.00,69.0,7.5,NaN,0.01
16255,Dungeons 2,PS4,2016,Role-Playing,0.01,0.00,0.00,0.00,61.0,7.9,T,0.01


In [55]:
critic_corr = ps4_critic['critic_score'].corr(ps4_critic['total_sales'])
user_corr = ps4_user['user_score'].corr(ps4_user['total_sales'])

print(f"Critic score and total sales correlation: {critic_corr:.4f}")
print(f"User score and total sales correlation: {user_corr:.4f}")

Critic score and total sales correlation: 0.4066
User score and total sales correlation: -0.0320


There is a moderate positive correlation between critic_score and total_sales (0.4066), while user_score shows almost no correlation (−0.0320).

Games with higher critic scores tend to sell slightly more, but user_score is not useful for predicting sales.

> Correlation is not Causation This does not imply causation. Sales may also be influenced by factors such as marketing, budget, and game quality.

In [56]:
fig_scatter_critic = px.scatter(ps4_critic, x='critic_score', y='total_sales',
                  title=f'PS4: Critic score vs Total sales (korrelation = {critic_corr:.4f})',
                  labels={'critic_score': 'Critic score', 'total_sales': 'Total sales'})
fig_scatter_critic.show()

In [57]:
fig_scatter_user = px.scatter(ps4_user, x='user_score', y='total_sales',
                  title=f'PS4: User score vs Total sales (korrelation = {user_corr:.4f})',
                  labels={'user_score': 'User score', 'total_sales': 'Total sales'})
fig_scatter_user.show()

## 3.7 Cross-Platform Comparison

In [58]:
result

,Name,count_of_name,platforms
6715,Need for Speed: Most Wanted,12,"DS, GBA, GC, PC, PS2, PS3, PSV, WiiU, X360, XB"
2952,FIFA 14,9,"3DS, PC, PS3, PS4, PSP, PSV, Wii, X360, XOne"
7785,Ratatouille,9,"DS, GBA, GC, PC, PS2, PS3, PSP, Wii, X360"
5143,LEGO Marvel Super Heroes,9,"3DS, DS, PC, PS3, PS4, PSV, WiiU, X360, XOne"
5470,Madden NFL 07,9,"DS, GBA, GC, PS2, PS3, PSP, Wii, X360, XB"
...,...,...,...
2837,Escape The Museum,2,"DS, Wii"
2846,Eternal Sonata,2,"PS3, X360"
2850,Etrian Odyssey II: Heroes of Lagaard,2,"3DS, DS"
2860,Evangelion: Jo,2,"PS2, PSP"


In [59]:
multi = games.groupby('name')['platform'].nunique()
multi = multi[multi>1].index

print(f'The games on more than 1 platfrom: {len(multi)}')

The games on more than 1 platfrom: 2748


In [60]:
name_abbreviations = {
    'Grand Theft Auto V': 'GTA V',
    'Call of Duty: Ghosts': 'CoD: Ghosts',
    'Call of Duty: Advanced Warfare': 'CoD: AW',
    'Call of Duty: Black Ops 3': 'CoD: BO3',
    "Assassin's Creed IV: Black Flag": "AC IV",
    'FIFA 14': 'FIFA 14',
    'FIFA 15': 'FIFA 15',
    'FIFA 16': 'FIFA 16',
    'Minecraft': 'Minecraft',
    'Battlefield 4': 'BF4'
}

In [61]:
multi_games = games_relevant[games_relevant['name'].isin(multi)]
top_10_multi_games = multi_games.groupby('name')['total_sales'].sum().sort_values(ascending=False).head(10).index

comparison = multi_games[multi_games['name'].isin(top_10_multi_games)].copy()
comparison['name_short'] = comparison['name'].replace(name_abbreviations)

fig_multi = px.bar(comparison, x='name_short', y='total_sales', color='platform', barmode='group',
                    title='Sales of the best-selling multi-platform games by platform',
                    labels={'name_short': 'Games', 'total_sales': 'Total sales', 'platform': 'Platform'})
fig_multi.update_xaxes(tickangle=45)
fig_multi.update_layout(width=1100, height=650)
fig_multi.show()

## 3.8 Genre Analysis

In [76]:
genre_sales = games_relevant.groupby('genre')['total_sales'].agg(total = 'sum',avg = 'mean',count_of_games = 'count').sort_values('total', ascending= False)
genre_sales['market_share_percent'] = (genre_sales['total'] / genre_sales['total'].sum() * 100).round(2)

genre_sales.round(2)

,total,avg,count_of_games,market_share_percent
genre,,,,
Action,321.87,0.42,766,29.51
Shooter,232.98,1.25,187,21.36
Sports,150.65,0.70,214,13.81
Role-Playing,145.89,0.50,292,13.38
Misc,62.82,0.41,155,5.76
Platform,42.63,0.58,74,3.91
Racing,39.89,0.47,85,3.66
Fighting,35.31,0.44,80,3.24
Adventure,23.64,0.10,245,2.17


In [78]:
fig_genre_sales = px.bar(genre_sales.reset_index().sort_values('total'), x='total', y='genre', orientation='h',
             title='Total sales by genre 2013-2016',
             labels={'genre': 'Genre', 'total': 'Total sales', 'market_share_percent': 'Market share (%)'},
             hover_data={'market_share_percent': ':.2f'})
fig_genre_sales.show()

In [79]:
fig = px.funnel(genre_sales.reset_index(), x='total', y='genre',
                 title='Genre sales ranking (funnel view)',
                 labels={'genre': 'Genre', 'total': 'Total sales', 'market_share_percent': 'Market share (%)'},
                 hover_data={'market_share_percent': ':.2f'})
fig.show()

The best-selling genre is `Action` (`321.87`, `~29-30%` market share), followed by `Shooter` (`232.98`), `sports` (`150.65`), and `Role-Playing` (`145.89`). `Simulation`, `Strategy` and `Puzzle` have the lowest sales.

`Action` and `Shooter` lead due to both high release volume and strong sales per game, while `Strategy/Puzzle` serve smaller niche audiences.

# Step 4: Regional Market Analysis and User Profiles

## 4.1 Regional Platform Analysis

In [65]:
games

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,total_sales
0,Wii Sports,Wii,2006,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E,82.54
1,Super Mario Bros.,NES,1985,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN,40.24
2,Mario Kart Wii,Wii,2008,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E,35.52
3,Wii Sports Resort,Wii,2009,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E,32.77
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN,31.38
...,...,...,...,...,...,...,...,...,...,...,...,...
16439,Samurai Warriors: Sanada Maru,PS3,2016,Action,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01
16440,LMA Manager 2007,X360,2006,Sports,0.00,0.01,0.00,0.00,NaN,NaN,NaN,0.01
16441,Haitaka no Psychedelica,PSV,2016,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN,0.01
16442,Spirits & Spells,GBA,2003,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN,0.01


In [66]:
def top_5_columns( data, region_column , group_column, n = 5 ):
    grouped = data.groupby(group_column)[region_column].sum().sort_values(ascending = False).head(n)
    total = data[region_column].sum()
    result1 = grouped.reset_index()
    result1.columns = [group_column,'sales']
    result1['market_share_percent'] = round(result1['sales']/total*100,2)
    return result1

regions = {'na_sales': 'North America', 'eu_sales': 'Europe', 'jp_sales': 'Japan'}

In [67]:
for col, region_name in regions.items():
    print(f"--- {region_name} ---")
    display(top_5_columns(games_relevant, col, 'platform'))

--- North America ---


,platform,sales,market_share_percent
0,PS4,108.74,24.84
1,XOne,93.12,21.27
2,X360,81.66,18.66
3,PS3,63.50,14.51
4,3DS,38.20,8.73


--- Europe ---


,platform,sales,market_share_percent
0,PS4,141.09,35.97
1,PS3,67.81,17.29
2,XOne,51.59,13.15
3,X360,42.52,10.84
4,3DS,30.96,7.89


--- Japan ---


,platform,sales,market_share_percent
0,3DS,67.81,48.17
1,PS3,23.35,16.59
2,PSV,18.59,13.21
3,PS4,15.96,11.34
4,WiiU,10.88,7.73


### Cross-Regional Platform Comparison

In [68]:
platform_regional = games_relevant.groupby('platform')[['na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'total_sales']].sum()
platform_regional = platform_regional.sort_values('total_sales', ascending=False)

for col in ['na_sales', 'eu_sales', 'jp_sales']:
    platform_regional[f'{col}_share_pct'] = (platform_regional[col] / platform_regional[col].sum() * 100).round(2)

platform_regional.round(2)

,na_sales,eu_sales,jp_sales,other_sales,total_sales,na_sales_share_pct,eu_sales_share_pct,jp_sales_share_pct
platform,,,,,,,,
PS4,108.74,141.09,15.96,48.35,314.14,24.84,35.97,11.34
PS3,63.50,67.81,23.35,26.77,181.43,14.51,17.29,16.59
XOne,93.12,51.59,0.34,14.27,159.32,21.27,13.15,0.24
3DS,38.20,30.96,67.81,6.28,143.25,8.73,7.89,48.17
X360,81.66,42.52,0.51,12.11,136.80,18.66,10.84,0.36
WiiU,29.21,19.85,10.88,4.69,64.63,6.67,5.06,7.73
PC,11.11,25.36,0.00,2.96,39.43,2.54,6.47,0.00
PSV,5.04,6.10,18.59,3.26,32.99,1.15,1.56,13.21
Wii,6.56,5.93,0.05,1.12,13.66,1.50,1.51,0.04


In [69]:
top_platforms = platform_regional.head(8).index.tolist()

regional_long = games[games['platform'].isin(top_platforms)].groupby('platform')[
    ['na_sales', 'eu_sales', 'jp_sales']
].sum().reset_index()

regional_long = regional_long.melt(id_vars='platform', var_name='region', value_name='sales')
regional_long['region'] = regional_long['region'].map({
    'na_sales': 'North America', 'eu_sales': 'Europe', 'jp_sales': 'Japan'
})

fig = px.bar(regional_long, x='platform', y='sales', color='region', barmode='group',
             category_orders={'platform': top_platforms},
             title='Comparison of Top 8 platfroms by sales (1980-2016)',
             labels={'platform': 'Platform', 'sales': 'Sales ', 'region': 'Region'})
fig.show()

## 4.2 Regional Genre Analysis

In [70]:
for col, region_name in regions.items():
    print(f"--- {region_name} ---")
    display(top_5_columns(games_relevant, col, 'genre'))

--- North America ---


,genre,sales,market_share_percent
0,Action,126.05,28.80
1,Shooter,109.74,25.07
2,Sports,65.27,14.91
3,Role-Playing,46.40,10.60
4,Misc,27.49,6.28


--- Europe ---


,genre,sales,market_share_percent
0,Action,118.13,30.12
1,Shooter,87.86,22.40
2,Sports,60.52,15.43
3,Role-Playing,36.97,9.43
4,Racing,20.19,5.15


--- Japan ---


,genre,sales,market_share_percent
0,Role-Playing,51.04,36.26
1,Action,40.49,28.76
2,Misc,9.20,6.54
3,Fighting,7.65,5.43
4,Shooter,6.61,4.70


NA & EU: `Action` leads, followed by `Shooter` and `Sports` (NA: 28.80% / 25.07% / 14.91%; EU: 30.12% / 22.40% / 15.43%) — both Western regions show near-identical genre preferences.
JP: `Role-Playing` is the top genre (36.26%), followed by `Action` (28.76%), while `Shooter` ranks far lower (4.70%).

> Western markets have similar preferences, while Japan shows a stronger preference for RPGs, consistent with its JRPG tradition.

## 4.3 ESRB Rating Impact Analysis

In [71]:
esrb_region = games_relevant.groupby('rating')[['na_sales', 'eu_sales', 'jp_sales','other_sales']].sum().round(2)
esrb_region = esrb_region.loc[esrb_region.sum(axis=1).sort_values(ascending=False).index]
esrb_region

,na_sales,eu_sales,jp_sales,other_sales
rating,,,,
M,165.21,145.32,14.11,47.04
E,79.05,83.36,15.14,22.61
T,49.79,41.95,20.59,14.29
E10+,54.24,42.69,5.89,12.57


In [72]:
esrb_long = esrb_region.reset_index().melt(id_vars='rating', var_name='region', value_name='sales')

fig = px.bar(esrb_long, x='rating', y='sales', color='region', barmode='group',
             title='ESRB rating sales comparison by region (2013-2016)',
             labels={'rating': 'ESRB rating', 'sales': 'Sales (mln USD)', 'region': 'Region'})
fig.show()

`M` (Mature) rated games lead sales in NA (165.21M) and EU (145.32M), while Japan is led by `T` (Teen) rated games (20.59M), with `E` (15.14M) close behind and `M` last (14.11M).

> ESRB impact is region-specific: Western audiences lean toward mature content, while Japan's audience leans toward broader-appeal (`T`) titles — consistent with the Role-Playing genre preference seen in section 4.2.

# Step 5 : Hypothesis Tests

In [73]:
xone_scores = games_relevant[games_relevant['platform'] == 'XOne']['user_score'].dropna()
pc_scores = games_relevant[games_relevant['platform'] == 'PC']['user_score'].dropna()

print(f"Xbox One: n={len(xone_scores)}, mean ={xone_scores.mean():.3f}, std={xone_scores.std():.3f}")
print(f"PC:       n={len(pc_scores)}, mean ={pc_scores.mean():.3f}, std={pc_scores.std():.3f}")

alpha = 0.05
result_1 = scipy_stats.ttest_ind(xone_scores, pc_scores, equal_var=False)
print(f"\nt-statistic = {result_1.statistic:.3f}")
print(f"p-value = {result_1.pvalue:.4f}")

if result_1.pvalue < alpha:
    print(f"p-value ({result_1.pvalue:.4f}) < alpha ({alpha}) → H0 is rejected")
else:
    print(f"p-value ({result_1.pvalue:.4f}) >= alpha ({alpha}) → H0 is not rejected")

Xbox One: n=182, mean =6.521, std=1.381
PC:       n=155, mean =6.270, std=1.742

t-statistic = 1.452
p-value = 0.1476
p-value (0.1476) >= alpha (0.05) → H0 is not rejected


**H0 (Null Hypothesis):** There is no statistically significant difference between the mean `user_score` of Xbox One and PC platforms `(μ_XOne = μ_PC)`.

**H1 (Alternative Hypothesis):** There is a statistically significant difference between the mean `user_score` of Xbox One and PC platforms `(μ_XOne ≠ μ_PC)`.

In [74]:
action_scores = games_relevant[games_relevant['genre'] == 'Action']['user_score'].dropna()
sports_scores = games_relevant[games_relevant['genre'] == 'Sports']['user_score'].dropna()

print(f"Action: n={len(action_scores)}, mean={action_scores.mean():.3f}, std={action_scores.std():.3f}")
print(f"Sports: n={len(sports_scores)}, mean={sports_scores.mean():.3f}, std={sports_scores.std():.3f}")

result_2 = scipy_stats.ttest_ind(action_scores, sports_scores, equal_var=False)
print(f"\nt-statistic = {result_2.statistic:.3f}")
print(f"p-value = {result_2.pvalue:.2e}")

if result_2.pvalue < alpha:
    print(f"p-value ({result_2.pvalue:.2e}) < alpha ({alpha}) → H0 is rejected")
else:
    print(f"p-value ({result_2.pvalue:.2e}) >= alpha ({alpha}) → H0 is not rejected")

Action: n=389, mean=6.838, std=1.330
Sports: n=160, mean=5.238, std=1.783

t-statistic = 10.233
p-value = 1.45e-20
p-value (1.45e-20) < alpha (0.05) → H0 is rejected


In [80]:
fig = px.box(games_relevant[games_relevant['genre'].isin(['Action', 'Sports'])],
             x='genre', y='user_score', color='genre',
             title='Action vs Sports: spread of user score',
             labels={'genre': 'Genre', 'user_score': 'User score from 10'})
fig.show()


p-value ≈ 1.45×10⁻²⁰ is far below α = 0.05 → H0 is strongly rejected.

There is a highly significant difference between Action (mean ≈ 6.84) and Sports (mean ≈ 5.24) user scores. Action games receive higher user ratings on average. A possible explanation is that Sports games often follow annual roster-update cycles, while Action games tend to offer more varied and innovative content. This remains an interpretation, not proven causation.

# Step 6. Write a general conclusion


### Data Preparation

- Converted column names to lowercase and removed 2 completely empty rows.
- Replaced `tbd` in `user_score` with `NaN`, not `0`.
- Kept missing values in `critic_score`, `user_score`, and `rating` because ~40–50% missing data makes imputation unreliable (6,580 rows have all three missing at once).
- Removed 269 rows with missing `year_of_release` and converted it to `Int64`.
- Created `total_sales` as the sum of the 4 regional sales columns.

### Key Findings

1. **Platform lifecycle:**
    - Platforms stay active for a median of 6 years (mean ≈ 7.6 years). Using a 3-year inactivity threshold, 20 platforms (PS2, XB, GBA, GC, PS, N64, SNES, and others) are classified as discontinued by 2016. The 2013 launch of PS4/XOne marked a clear generation shift, which is why **2013–2016** was selected as the relevant period for the 2017 forecast.
2. **Leaders in the relevant period (2013–2016):**
    - `PS4` leads with 314.14M in sales, followed by `PS3` (181.43M), `XOne` (159.32M), `3DS` (143.25M), and `X360` (136.80M). PS4 and XOne show a growing trend, while PS3/X360/Wii/DS are declining as older-generation platforms.
3. **Reviews:**
    - For PS4, `critic_score` has a moderate positive correlation with sales (r ≈ 0.41), while `user_score` shows almost no correlation (r ≈ −0.03). Correlation ≠ causation — other factors like marketing and budget likely play a role.
4. **Genre (all-time, 1980–2016):**
    - `Action` leads with **≈19.5%** market share (1716.52M), followed by `Sports` (14.9%), `Shooter` (11.8%), and `Role-Playing` (10.6%). `Adventure`, `Strategy`, and `Puzzle` are the smallest niches.
5. **Regional differences (2013–2016 relevant period):**
   - *Platforms:* `PS4` is the sales leader in both NA (24.84% share) and EU (35.97% share), consistent with the overall platform leader found in point 2. Japan is dominated by `3DS` (48.17% share) — a clear outlier from the Western pattern, where handheld/local platforms are far stronger than in NA/EU.
   - *Genres:* NA and EU both favor `Action` > `Shooter` > `Sports` (NA: 28.80% / 25.07% / 14.91%; EU: 30.12% / 22.40% / 15.43%). Japan clearly favors `Role-Playing` (36.26% share) over `Action` (28.76%), with `Shooter` far behind (4.70%) — consistent with Japan's JRPG tradition.
   - *ESRB rating:* `M` (Mature) rated games lead sales in NA (165.21M) and EU (145.32M). Japan is the outlier again, led by `T` (Teen) rated games (20.59M) rather than `M` — aligned with Japan's stronger RPG/all-ages preference.
6. **Hypothesis tests:**
   - `XOne` vs. `PC` user scores: no statistically significant difference (p = 0.148 > 0.05) — H0 not rejected.
   - `Action` vs. `Sports` user scores: highly significant difference (p ≈ 1.45e−20 ≪ 0.05) — H0 strongly rejected. Action games are rated higher on average (6.84 vs. 5.24).

### 2017 Advertising Recommendations
- Focus advertising spend on `PS4` titles, and on the `Action`/`Shooter` genres in NA and EU — these show the strongest current sales and align with each region's top platform and genre preferences.
- Use a distinct strategy for Japan: prioritize `Role-Playing` titles on `3DS`, and lean toward broader-audience (`T`-rated) content rather than mature-only titles, since Japan's regional profile (platform, genre, and ESRB rating) diverges sharply from NA/EU on every dimension measured.
- Rely more on critic scores than user scores when using reviews as a marketing signal, since critic scores show a stronger (though still moderate) relationship with sales.
